In [1]:
import csv
import requests
import pandas
import folium


In [2]:
class Cart:
    """folium map with markers"""

    def __init__(self, location=None, zoom_start=5):
        self.map = folium.Map(location=location, zoom_start=zoom_start)

    def add_markers(self, location, **kwargs):
        """add markers with popup in a layer group"""
        opt = {
            "popup": None,
            "max_width": 150,
            "icon": "bomb",
            "color": "blue",
            "group": None,
        } | kwargs
        grp = self.map
        color = opt["color"]
        if opt["group"]:
            grp = folium.map.FeatureGroup(opt["group"])
            grp.add_to(self.map)
        if opt["popup"]:
            for loc, pop in zip(location, opt["popup"]):
                icon = folium.Icon(color=color, icon=opt["icon"], prefix="fa")
                txt = folium.Popup(Cart.html(pop), max_width=opt["max_width"])
                folium.Marker(location=loc, popup=txt, icon=icon).add_to(grp)
        else:
            for loc in location:
                icon = folium.Icon(color=color, icon=opt["icon"], prefix="fa")
                folium.Marker(location=loc, icon=icon).add_to(grp)

    def show(self, mapname=None):
        """return the map and save it if mapname is present"""
        folium.LayerControl().add_to(self.map)
        if mapname:
            self.map.save(mapname)
        return self.map

    @staticmethod
    def html(dic):
        """return an html string from a dict"""
        pop = ""
        for key, val in dic.items():
            pop += "<div>" + "<b>" + str(key) + "</b>" + " : " + str(val) + "</div>"
        return pop


In [4]:
# lire le fichier csv et créer un DataFrame pandas
terrains = pandas.read_csv('prix_terrains.csv')
terrains['prix_m2'] = terrains['prix total'] / terrains['surface']
terrains.columns


Index(['Commune', 'Latitude', 'Longitude', 'surface', 'prix total', 'prix/m2',
       'statut', 'Type', 'Nom', 'commentaires', 'Unnamed: 10', 'Unnamed: 11',
       'prix_m2'],
      dtype='object')

In [5]:
terrains

,Commune,Latitude,Longitude,surface,prix total,prix/m2,statut,Type,Nom,commentaires,Unnamed: 10,Unnamed: 11,prix_m2
0,Amigny Rouy (02700),49.607274,3.308428,5427,82000.0,15.0,Constructible,Agence,Devos Jonathan,NaN,NaN,15.109637,15.109637
1,Amigny-Rouy (02700),49.607274,3.308428,882,31000.0,35.0,Constructible,Agence,Challenge Immobilier,NaN,NaN,NaN,35.147392
2,Appilly (60400),49.582919,3.116574,678,44000.0,64.0,Constructible,Agence,CABINET CERI,NaN,NaN,NaN,64.896755
3,Attichy (60350),49.412110,3.048988,466,41000.0,87.0,Constructible,Notaire,Me Emmanuel FRANCOIS,NaN,NaN,NaN,87.982833
4,Attichy (60350),49.412110,3.048988,2132,18000.0,8.0,Non constructible,Agence,ERA IMMOBILIER,NaN,NaN,NaN,8.442777
...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,Trosly-Loire (02300),49.517113,3.227923,1085,39900.0,37.0,Constructible,Agence,NADEGE BOMPIS\nQUENTIMMO CHAUNY Nadège BOMPIS ...,"environ 29m de façade pour 40m de longueur, av...",NaN,NaN,36.774194
265,Trosly-Loire (02300),49.517113,3.227923,763,20000.0,26.0,Constructible,Particulier,Particulier sur bon coin,entièrement clôturé avec portail. Terrain enti...,NaN,NaN,26.212320
266,Vézaponin (02290),49.458666,3.230062,1600,45000.0,28.0,Constructible,Particulier,Particulier sur bon coin,NaN,NaN,NaN,28.125000
267,Vic sur Aisne (02290),49.405133,3.112714,1390,56000.0,40.0,Constructible,Agence,CENTURY 21 LS IMMOBILIER,NaN,NaN,NaN,40.287770


In [36]:
# création de la carte
map_terrains = folium.Map(location=(49.5198282,3.0259315),zoom_start=13)
constructible = folium.map.FeatureGroup("constructible")
non_constructible = folium.map.FeatureGroup("Non constructible")
constructible_viabilise = folium.map.FeatureGroup("Constructible viabilisé")

In [37]:
for index, row in terrains.iterrows():
    popup_html = f"""
    <b>commune :</b> {row['Commune']}<br>
    <b>surface :</b> {row['surface']} m2<br>
    <b>prix :</b> {row['prix total']} €<br>
    <b>prix/m2 :</b> {row['prix_m2']} €/m2<br>
    <b>statut :</b> {row['statut']}<br>
    <b>type :</b> {row['Type']}<br>
    """
# Ajout du marqueur
# couleyurs : {'pink', 'cadetblue', 'red', 'white', 'lightred', 'beige', 'lightblue', 'black', 'gray', 'darkred', 'blue', 'green', 'purple', 'darkgreen', 'darkblue', 'darkpurple', 'orange', 'lightgreen', 'lightgray'}.
    quintile = row['prix_m2']
    if quintile < 26 :
      color = 'gray'
    elif quintile < 42:
      color = 'green'
    elif quintile < 69:
      color = 'blue'
    elif quintile < 101:
      color = 'purple'
    else :
      color = 'red'
            
    marker = folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=str(row['surface']) +' m2',
        icon = folium.Icon(color=color)
    )
    match row['statut']:
        case "Constructible":
            marker.add_to(constructible)
        case "Non constructible":
            marker.add_to(non_constructible)
        case "Constructible viabilisé":
            marker.add_to(constructible_viabilise)

constructible_viabilise.add_to(map_terrains)
constructible.add_to(map_terrains)
non_constructible.add_to(map_terrains)

# Define the legend's HTML
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 600px; height: 130px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.85;">
     &nbsp; <b>Quintiles des prix au m2 (5 catégories de 20%) </b> <br>
     &nbsp; Q1 < 26€ Très peu cher - souvent grandes parcelles, zones rurales ou non viabilisées  &nbsp; <i class="fa fa-circle" style="color:gray"></i><br>
     &nbsp; Q2 < 42€ Bas de gamme - Typique des communes rurales avec parcelles de grande taille  &nbsp; <i class="fa fa-circle" style="color:green"></i><br>
     &nbsp; Q3 < 69€ Gamme moyenne - Beaucoup de terrains constructibles "standard"               &nbsp; <i class="fa fa-circle" style="color:blue"></i><br>
     &nbsp; Q4 < 101€ Haut de gamme - Petites parcelles bien situées ou partiellement viabilisées  &nbsp; <i class="fa fa-circle" style="color:purple"></i><br>
     &nbsp; Q5 > 101 Premium - lots viabilisés, surfaces plus petites, emplacements recherchés    &nbsp; <i class="fa fa-circle" style="color:red"></i><br>
</div>
'''

# Add the legend to the map
map_terrains.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(map_terrains)

map_terrains

In [38]:
# Sauvegarder la carte dans un fichier HTML
map_terrains.save("map_terrains.html")